# Quant Alpha Research: Comparing XGBoost, Transformers, HMM Regimes, and Reinforcement Learning

This project compares classical machine learning methods, transformer-based sequence models, hidden Markov model regime detection, and reinforcement learning agents for detecting and trading market alpha.

The goal is to evaluate whether deep sequence models, regime-aware features, and reinforcement-learning-based decision policies can outperform simpler baselines such as XGBoost under realistic backtesting conditions.

The first version of the project focuses on NVDA as the primary trading target. Additional market and sector assets such as QQQ, SPY are included as context features to help distinguish NVDA-specific alpha from broader market.

---

## Research Objective

The main research question is:

**Can transformer models, HMM-based regime detection, and reinforcement learning improve trading performance compared to a simpler XGBoost baseline?**

The project is not assuming that transformers, HMMs, or reinforcement learning are automatically better. Instead, the goal is to test whether each added layer of complexity produces measurable improvement over a strong classical machine learning baseline.

The comparison will be based on:

- Predictive accuracy
- Classification quality
- Probability calibration
- Trading PnL
- Sharpe ratio
- Maximum drawdown
- Turnover
- Transaction-cost-adjusted returns
- Out-of-sample performance

The project will also test whether HMM regime features provide real value or simply add noise and overfitting.

---

## Assets

### Primary Trading Target

- NVDA

### Context Assets

- QQQ
- SPY

NVDA is used as the first trading target because it is liquid, volatile, and strongly connected to semiconductor and AI-related market movements.

The context assets are used to help the models understand whether NVDA movement is stock-specific, sector-driven, or market-wide.

---

## Data Source and Trading Timeline

The first version of this project will use the Alpaca SIP feed to collect market data.

The base trading setup uses:

- Data source: Alpaca SIP feed
- Candle size: 5-minute candles
- Primary target: NVDA
- Context assets: QQQ, SPY
- Prediction horizon: 12 candles
- Prediction window: 60 minutes
- Trading style: intraday
- Overnight positions: disabled
- Trading day start: 6:30 AM Pacific Time
- Daily episode reset: enabled
- EMA warmup: 21 pre-open candles

Each model receives information available up to the current 5-minute candle and predicts the probability of a meaningful price move over the next 12 candles.

---

## Prediction Target

The first version frames alpha detection as a probabilistic classification problem.

At each 5-minute candle, the model predicts whether NVDA will move more than +1% or -1% within the next 12 candles, equivalent to the next 60 minutes.

For each candle at time `t`, the model outputs:

- `P(up)`: probability that NVDA hits the +1% upper barrier within the next 12 candles
- `P(down)`: probability that NVDA hits the -1% lower barrier within the next 12 candles
- `P(neutral)`: probability that neither barrier is reached within the next 12 candles

This creates a triple-barrier classification setup:

- Upper barrier: +1%
- Lower barrier: -1%
- Time barrier: 12 candles
- Base timeframe: 5-minute candles

If both the upper and lower barriers are reached within the 12-candle window, the label is assigned based on which barrier is reached first.

To avoid lookahead bias, features at candle `t` are only allowed to use information available at or before the close of candle `t`. Trading decisions are evaluated from the next candle onward.

---

## Data Cleaning and Missing Candle Filtering

Before labeling, training, or backtesting, the data pipeline will validate each trading day.

Because the project uses synchronized 5-minute candles across NVDA and all context assets, missing candles can create bad features, incorrect labels, and misleading backtest results.

The first version will use strict full-session filtering.

Only days with 100% complete data for every required asset will be kept. This means a trading day is valid only if NVDA and every context asset have all required 5-minute candles for the regular trading session.

The required assets are:

- NVDA
- QQQ
- SPY
- SMH
- AMD
- TSM
- AVGO
- MSFT

The first version will use the following filtering rules:

- Keep only days with complete regular-session NVDA candles
- Keep only days where every context asset has 100% of the required regular-session candles
- Require aligned timestamps across NVDA and all context assets
- Drop any day where any required asset is missing even one regular-session candle
- Drop rows where the 12-candle future labeling window is incomplete
- Exclude the final 12 candles of each day from new prediction labels
- Use warmup candles only for indicator calculation
- Do not use warmup candles for trading rewards or supervised labels

A normal full trading day contains 78 five-minute candles:

- 6.5 trading hours = 390 minutes
- 390 minutes / 5 minutes = 78 candles

For a day to be included, each required asset must have all 78 regular-session 5-minute candles aligned to the same timestamps.

Since the prediction horizon is 12 candles, the final 12 candles of each day cannot be used for new supervised labels unless the project explicitly allows the prediction window to continue beyond the close.

The first version will not allow prediction windows to continue beyond the close.

This keeps the labeling and backtesting logic clean:

- No missing asset data
- No partially aligned trading days
- No overnight labels
- No overnight positions
- No future window that extends past the trading session
- No training rows with incomplete future information

---

## Daily Reset and EMA Warmup

Each trading day is treated as a separate trading episode.

At the beginning of each day, the trading state is reset:

- Current position is reset to flat
- Entry price is reset
- Unrealized PnL is reset
- Bars in position is reset
- Signal-state counters are reset
- No overnight position is carried into the next day

However, technical indicators are not started from zero at the market open.

To avoid inaccurate EMA values at the beginning of the session, the data pipeline will load 21 additional 5-minute candles before the regular market open. These pre-open candles are used only to warm up indicators such as EMA 9 and EMA 20.

The warmup setup is:

- Warmup candles before open: 21
- Warmup duration: 105 minutes
- Regular market open: 6:30 AM Pacific Time
- Trading begins: 6:30 AM Pacific Time
- Overnight positions: disabled

The agent is not allowed to trade during the warmup period. Warmup candles are used only for feature calculation.

---

## Feature Set

For a fair comparison, XGBoost and the transformer model will be trained using the same core market information.

The difference is representation:

- XGBoost receives the features as a tabular row at each candle.
- The transformer receives a sequence of feature rows over a historical lookback window.

This allows the project to test whether the transformer architecture improves performance when both models are given access to the same information.

---

## Feature Normalization and Stationarity

Raw prices will not be used directly as model inputs because NVDA's price level can change significantly over a multi-year period.

Instead, the feature pipeline will focus on normalized and stationary features.

The model will prefer features such as:

- Log returns
- Percentage returns
- Price-normalized candle ranges
- Price-normalized wick sizes
- Price-normalized EMA distances
- Relative volume
- Volume z-scores
- Rolling volatility
- Cross-asset relative returns

Example normalized features include:

- `log_return_1 = log(close_t / close_t-1)`
- `ema_9_distance = (close - ema_9) / close`
- `ema_20_distance = (close - ema_20) / close`
- `ema_spread = (ema_9 - ema_20) / close`
- `candle_range_pct = (high - low) / close`
- `body_pct = (close - open) / open`
- `relative_volume = volume / rolling_average_volume`
- `nvda_minus_qqq_return = nvda_return - qqq_return`
- `nvda_minus_smh_return = nvda_return - smh_return`

Using normalized features helps prevent the model from memorizing historical price levels and makes the features more comparable across different market regimes.

If scalers are used, they must be fit only on the training set. Validation and test data must be transformed using the training-fitted scaler to avoid data leakage.

---

## Candle and Return Features

The first feature set will include price and candle-structure information such as:

- Lagged returns
- Rolling returns
- Log returns
- Candle body
- Candle range
- Upper wick
- Lower wick
- Close position within the candle range
- Rolling high-low range
- Rolling volatility

---

## EMA Features

The first technical indicator set will include EMA-based features such as:

- EMA 9
- EMA 20
- EMA 9 minus EMA 20
- EMA 9 above EMA 20
- EMA cross up
- EMA cross down
- Bars since last EMA cross
- EMA 9 slope
- EMA 20 slope

These features help the models detect short-term trend shifts, momentum, and possible regime changes.

To avoid inaccurate EMA values at the market open, the feature pipeline will use 21 pre-open 5-minute candles as a warmup window. These warmup candles are used only for indicator calculation and are not used for trading rewards or supervised labels.

---

## Volume Features

Volume features will include:

- 5-minute volume
- Rolling average volume
- Rolling volume standard deviation
- Relative volume
- Volume z-score
- Volume change
- Volume momentum

These features help identify whether a price move is supported by unusual activity or occurring on weak volume.

---

## Context Asset Features

The models will also use selected features from context assets:

- QQQ
- SPY

Context features may include:

- Asset returns
- Rolling returns
- Rolling volatility
- Relative movement versus NVDA
- Sector-relative strength
- Market-relative strength

These features help separate NVDA-specific movement from broader market and semiconductor-sector movement.

---

## HMM Regime Features

A hidden Markov model will be tested as a regime detection layer.

The HMM will not replace XGBoost, the transformer, or the RL agent. Instead, it will generate regime features that may help the models understand whether the market is currently trending, choppy, volatile, quiet, risk-on, or risk-off.

The HMM may use stationary market features such as:

- NVDA returns
- Rolling volatility
- Relative volume
- Candle range percentage
- NVDA return minus QQQ return
- NVDA return minus SMH return
- SPY return
- QQQ return
- SMH return

The HMM may generate features such as:

- `hmm_regime`
- `hmm_regime_prob_0`
- `hmm_regime_prob_1`
- `hmm_regime_prob_2`
- `regime_changed`
- `bars_since_regime_change`

The first version will likely test 3 regimes, then compare against 2, 4, and 5 regimes later.

To avoid data leakage, the HMM must be trained only on the training set. The fitted HMM is then used to infer regimes on validation and test data without refitting on future data.

---

## Point of Control Features

Point of Control features may be added later if sufficient intraday volume-at-price or trade-level data is available.

With standard 5-minute OHLCV candles, true Point of Control cannot be calculated directly because OHLCV data does not show how volume was distributed across price levels inside each candle.

If only candle data is available, the project may use approximate volume-price features, but true POC will be treated as a later extension.

---

## Class Imbalance

The triple-barrier labels may be imbalanced because most 60-minute windows may not hit either the +1% or -1% barrier.

For example, many candles may be labeled `neutral`, while fewer candles may be labeled `up` or `down`.

Accuracy alone will not be treated as sufficient because a model could achieve high accuracy by overpredicting the majority class, especially if `neutral` dominates.

The project will track the distribution of `up`, `down`, and `neutral` labels across every training, validation, and test period.

The model will be evaluated using:

- Confusion matrix
- Per-class precision
- Per-class recall
- Per-class F1 score
- Macro F1 score
- Log loss
- Brier score

This is important because the tradable events are the `up` and `down` classes. A model that predicts `neutral` well but fails to identify `up` or `down` events is not useful for trading.

If needed, class weights, balanced sampling, or probability threshold tuning may be used. These choices will be selected using only training and validation data.

---

## Probability Calibration

Because the models output probabilities, the project will evaluate not only whether the predicted class is correct, but also whether the predicted probabilities are reliable.

A model that predicts `P(up) = 0.70` should ideally be correct around 70% of the time when making similar predictions.

The project may evaluate calibration using:

- Brier score
- Log loss
- Reliability curves
- Confidence bucket analysis
- Expected calibration error

This is important because the trading layer depends on whether the probabilities are trustworthy.

A poorly calibrated model may still classify correctly sometimes, but its probability values may be misleading for position sizing, threshold selection, and reinforcement learning.

---

## Methods

### 1. XGBoost Baseline

XGBoost is used as the classical machine learning baseline.

The XGBoost model will use engineered market features such as candle features, EMA features, volume features, cross-asset features, HMM regime features when enabled, and time-of-day features.

The model outputs:

- `P(up)`
- `P(down)`
- `P(neutral)`

This gives a strong and interpretable baseline that the transformer, HMM, and RL systems must beat.

---

### 2. Transformer-Based Predictor

The transformer model will learn from sequences of historical 5-minute candles.

For a fair comparison, the transformer will use the same core feature set as the XGBoost baseline, including price features, EMA features, volume features, cross-asset features, HMM regime features when enabled, and time-of-day features.

The difference is that XGBoost receives these features as a single tabular row at each candle, while the transformer receives a sequence of feature rows over a lookback window.

The goal is to test whether the transformer can use temporal structure and cross-asset relationships more effectively than XGBoost when both models are given access to the same market information.

The transformer outputs:

- `P(up)`
- `P(down)`
- `P(neutral)`

---

### 3. HMM Regime Detection

The hidden Markov model is used as an optional regime feature generator.

The purpose of the HMM layer is to test whether regime awareness improves prediction or trading performance. For example, the same bullish model probability may be more useful in a high-volatility breakout regime than in a choppy mean-reverting regime.

The HMM will be evaluated by comparing model performance with and without HMM regime features.

The HMM features may be used by:

- XGBoost
- Transformer
- Rule-based trading systems
- Reinforcement learning agents

The goal is to answer:

**Does adding HMM regime information improve out-of-sample prediction and trading performance?**

---

### 4. Signal-State Features

After the prediction model outputs probabilities, the trading layer creates additional signal-state features.

Examples include:

- `p_up`
- `p_down`
- `p_neutral`
- `p_edge = p_up - p_down`
- `p_abs_edge`
- `signal_direction`
- `signal_age_bars`
- `signal_active`
- `p_edge_change_1`
- `p_edge_rolling_mean`
- `signal_persistence`

These features help the trading system understand not only the current prediction, but also how strong, persistent, and fresh the signal is.

For example, a fresh bullish signal may be treated differently from a bullish signal that has already been active for several candles.

---

### 5. Rule-Based Trading Layer

Before using reinforcement learning, the project will test simple rule-based trading policies.

Example baseline rules may include:

- Go long when `P(up)` is high enough and `P(up) - P(down)` exceeds a minimum edge
- Go short when `P(down)` is high enough and `P(down) - P(up)` exceeds a minimum edge
- Stay flat when confidence is weak or the probabilities are too close

Thresholds will be selected only using training and validation data.

The final test set will not be used for threshold tuning.

This rule-based layer provides a simple trading baseline before evaluating whether reinforcement learning improves trade timing, exits, and risk management.

---

### 6. Reinforcement Learning Trading Agent

The reinforcement learning agent is used as the trading decision layer.

The RL agent does not replace the prediction model. Instead, it receives model probabilities, signal-state features, HMM regime features, market features, and portfolio state features, then learns how to act on them.

The RL observation/state may include:

#### Market Features

- NVDA return features
- NVDA volatility
- NVDA volume
- Candle body and range
- EMA features
- Volume features
- QQQ returns
- SPY returns

#### Model Signal Features

- `p_up`
- `p_down`
- `p_neutral`
- `p_edge`
- `signal_direction`
- `signal_age_bars`
- `signal_persistence`
- Probability momentum features

#### HMM Regime Features

- `hmm_regime`
- HMM regime probabilities
- Regime change flag
- Bars since regime change

#### Trading State Features

- Current position
- Bars in position
- Entry price
- Unrealized PnL
- Max favorable excursion
- Max adverse excursion
- Time of day
- Remaining bars until market close

The purpose of the RL layer is to learn trade timing, position management, exits, and risk control.

---

## RL Episode Structure

Each RL episode represents one trading day.

At the beginning of each day, the agent starts flat and all portfolio state variables are reset.

The trading state reset includes:

- Current position reset to flat
- Entry price reset
- Unrealized PnL reset
- Bars in position reset
- Signal-state counters reset
- No overnight position carried into the next day

The feature builder has access to 21 pre-open 5-minute warmup candles for technical indicators such as EMA 9 and EMA 20.

The RL agent does not trade during the warmup period. Warmup candles exist only to initialize indicators.

The episode terminates at the end of the trading day. Any open position is closed before the episode ends to enforce the no-overnight-position rule.

---

## RL Action Space

The first RL environment will use a simple discrete action space.

The agent chooses the target position for the next candle:

- `0`: short
- `1`: flat
- `2`: long

Later versions may test continuous position sizing, where the agent chooses a target exposure between full short and full long.

---

## RL Reward Function

The reinforcement learning agent will be rewarded based on transaction-cost-adjusted portfolio value changes.

The reward may include:

- Realized PnL
- Unrealized PnL
- Slippage and fees
- Penalty for excessive turnover
- Penalty for large drawdowns
- Penalty for holding positions too close to market close
- Optional risk-adjusted reward terms

The first reward function will prioritize simple net PnL after costs.

More complex reward shaping may be added later only after the baseline RL environment is working.

---

## Execution Timing

To avoid lookahead bias, the model can only use information available at or before the close of candle `t`.

If a signal appears on candle `t`, the strategy cannot execute inside candle `t`.

The earliest possible execution is the open of candle `t+1`.

The default execution assumption is:

- Features are observed through candle `t`
- Prediction is generated after candle `t` closes
- Signal is generated after candle `t` closes
- Trade is executed at the open of candle `t+1`
- Label window is evaluated over the next 12 candles

This prevents the model from using the close, high, low, or volume of candle `t` and pretending it could have traded earlier inside that same candle.

The trading rule is:

- Signal at candle `t` executes at candle `t+1` open

This applies to all systems:

- XGBoost probabilities + rule-based trading
- Transformer probabilities + rule-based trading
- XGBoost probabilities + RL trading agent
- Transformer probabilities + RL trading agent
- All HMM and non-HMM variants

---

## Costs and Slippage

Backtests will include estimated trading costs.

The first version will model:

- Commission or fee per trade
- Slippage in basis points
- Optional spread cost
- Position sizing constraints
- Turnover

A simple first cost model is:

- Long entry price = next open adjusted upward by slippage
- Long exit price = next open adjusted downward by slippage
- Short entry price = next open adjusted downward by slippage
- Short exit price = next open adjusted upward by slippage

This makes the backtest more conservative and prevents strategies from appearing profitable only because they ignore execution costs.

The project will compare results before and after costs, but the main reported performance will be transaction-cost-adjusted.

---

## Walk-Forward Validation and Final Test Set

The project will use walk-forward validation with a locked final test set.

The first version assumes a 10-year historical dataset.

The split will be:

- Years 1-3: Train fold 1
- Year 4: Validate fold 1
- Years 5-7: Train fold 2
- Year 8: Validate fold 2
- Years 9-10: Final out-of-sample test

The first two folds are used for model development, validation, threshold tuning, HMM regime selection, RL tuning, and feature selection.

The final two years are held out and used only once for final evaluation after the methodology is locked.

The final test set will not be used for:

- Feature selection
- Hyperparameter tuning
- Probability threshold tuning
- HMM regime count selection
- RL reward tuning
- RL policy selection
- Backtest rule changes

This ensures that the final reported results represent true out-of-sample performance rather than repeated tuning on the test period.

For each fold:

- Feature scalers are fit only on the training period
- HMM is fit only on the training period
- XGBoost and transformer models are trained only on the training period
- Validation is used for tuning and model selection
- Test data remains untouched until final evaluation

This validation structure is especially important because the project compares 8 systems and includes high-capacity models that can easily overfit.

---

## Model Selection and Hyperparameter Tuning

Hyperparameters and model choices will be selected using only training and validation periods.

The final test set will not be used for:

- Feature selection
- Hyperparameter tuning
- Probability threshold tuning
- HMM regime count selection
- RL reward tuning
- RL policy selection
- Backtest rule changes

This prevents test-set leakage and keeps the final out-of-sample evaluation meaningful.

---

## Non-ML Baselines

The project will compare machine learning systems against simple non-ML baselines.

Possible baselines include:

- Always flat
- Buy and hold NVDA
- EMA 9/20 crossover strategy
- Simple momentum strategy
- Simple mean-reversion strategy
- Random trading policy

These baselines help determine whether the ML systems are adding real value beyond simple market exposure or basic technical rules.

---

## Ablation Studies

The project will use ablation studies to identify which components actually improve performance.

Each major component will be tested by comparing model variants with and without that component.

Examples include:

- Without context assets
- With context assets
- Without EMA features
- With EMA features
- Without volume features
- With volume features
- Without HMM regime features
- With HMM regime features
- Rule-based trader versus RL trader
- XGBoost versus transformer predictor

Ablation studies help determine whether performance comes from the model architecture, feature engineering, regime detection, or the trading decision layer.

---

## Risk Management Constraints

The backtester and RL environment will include simple risk controls.

The first version may include:

- Maximum position size
- No overnight positions
- Forced flat before market close
- Maximum holding period
- Slippage and fee modeling
- Optional stop-loss and take-profit rules
- Optional maximum daily loss limit

These constraints make the trading environment more realistic and prevent the agent from exploiting unrealistic behavior.

---

## Model Comparison

The project will compare 8 main ML/RL systems.

### Without HMM Regime Features

1. XGBoost probabilities + simple rule-based trading
2. Transformer probabilities + simple rule-based trading
3. XGBoost probabilities + reinforcement learning trading agent
4. Transformer probabilities + reinforcement learning trading agent

### With HMM Regime Features

5. XGBoost + HMM regime features + simple rule-based trading
6. Transformer + HMM regime features + simple rule-based trading
7. XGBoost + HMM regime features + reinforcement learning trading agent
8. Transformer + HMM regime features + reinforcement learning trading agent

This structure allows the project to answer three separate questions:

1. Does the transformer improve prediction quality over XGBoost?
2. Does reinforcement learning improve trading decisions compared to simple rules?
3. Does HMM regime detection improve prediction and/or trading performance?

The HMM experiments are included to test whether regime detection provides real out-of-sample value or simply adds unnecessary complexity.

---

## Backtesting Assumptions

The backtester will simulate realistic intraday execution using 5-minute candles.

The first version will use simple but explicit assumptions:

- 5-minute candle execution
- Intraday trading only
- No overnight positions
- Daily episode reset at the regular market open
- Trading day starts at 6:30 AM Pacific Time
- Trading state resets at the start of each day
- EMA warmup uses 21 pre-open 5-minute candles
- No trading is allowed during the EMA warmup period
- No new trades are allowed when the full prediction or holding window is unavailable
- Signals generated on candle `t` can only execute at the open of candle `t+1`
- Transaction costs are included
- Slippage is included
- Optional spread cost is included
- Position limits are included
- Out-of-sample test data is separated from training and validation data
- HMM models are trained only on training data to avoid leakage

The backtester will evaluate both prediction quality and trading performance.

---

## Reproducibility

Experiments will track:

- Dataset version
- Feature configuration
- Model type
- Hyperparameters
- Random seed
- Train/validation/test split
- Backtest settings
- Transaction cost assumptions
- Performance metrics

The goal is for every result to be reproducible from a saved configuration file.

---

## Project Workflow

The notebook is used as the research interface, while the main logic lives in scripts and reusable modules.

The intended workflow is:

1. Download or load Alpaca SIP candle data
2. Align NVDA with context assets
3. Validate trading sessions and remove days with missing required candles
4. Keep only days with 100% complete data for all required assets
5. Load 21 pre-open warmup candles for each valid day
6. Build normalized market features using log returns, percentage returns, relative volume, and price-normalized indicators
7. Build EMA, volume, volatility, time-of-day, and cross-asset features
8. Build triple-barrier labels using the next 12 candles
9. Remove rows where the full 12-candle future label window is unavailable
10. Track class balance across train, validation, and test periods
11. Train HMM regime detector on training data only
12. Infer HMM regime features for train, validation, and test data
13. Train the XGBoost baseline with and without HMM features
14. Train the transformer predictor with and without HMM features
15. Generate prediction probabilities
16. Evaluate probability calibration
17. Build signal-state features
18. Tune rule-based trading thresholds on validation data only
19. Train or evaluate RL trading agents with and without HMM features
20. Run backtests with slippage, fees, spread assumptions, and position limits
21. Compare all 8 main ML/RL experiments in the notebook
22. Run ablation studies and non-ML baseline comparisons
23. Evaluate the locked final test set once after methodology is fixed

---

## Initial Development Plan

### Phase 1: Data Pipeline

Build a clean data loader for Alpaca SIP 5-minute candle data.

The data pipeline will:

- Align NVDA and context assets on the same timestamp index
- Validate each trading session
- Remove days with missing required candles
- Keep only days with 100% complete data for every required asset
- Preserve 21 pre-open warmup candles for indicator calculation
- Separate warmup candles from tradable candles

### Phase 2: Session Filtering and Warmup Handling

Implement strict session-level filtering.

Each valid day should include:

- Complete regular-session candles
- Required context asset candles aligned to NVDA
- 21 pre-open warmup candles when available
- No missing required candles in the tradable session

Warmup candles are used for feature calculation only and are excluded from trading, labeling, and RL rewards.

### Phase 3: Labeling

Create triple-barrier labels for whether NVDA hits +1%, -1%, or neither within the next 12 candles.

Rows without a complete 12-candle future window are removed from supervised training.

### Phase 4: Feature Engineering

Build normalized and stationary features, including:

- Log returns
- Percentage returns
- Price-normalized candle features
- EMA features
- Volume features
- Cross-asset features
- Time features
- Volatility features

### Phase 5: Class Balance and Calibration Checks

Analyze the distribution of `up`, `down`, and `neutral` labels.

Evaluate model outputs using per-class metrics and calibration metrics so the model does not appear successful by overpredicting the majority class.

### Phase 6: XGBoost Baseline

Train an XGBoost classifier to predict:

- `P(up)`
- `P(down)`
- `P(neutral)`

Run this model both with and without HMM regime features.

### Phase 7: HMM Regime Detection

Train an HMM regime detector using only the training set.

Generate regime features such as:

- Regime IDs
- Regime probabilities
- Regime change flags
- Bars since regime change

Compare XGBoost performance with and without HMM features before using HMM features in more complex models.

### Phase 8: Transformer Predictor

Train a transformer model on multi-asset candle sequences using the same core information as the XGBoost baseline.

Compare transformer performance with and without HMM regime features.

### Phase 9: Signal-State Features

Use model probabilities to create signal features such as:

- Signal direction
- Signal age
- Probability edge
- Probability momentum
- Signal persistence

### Phase 10: Rule-Based Trading Baseline

Build simple probability-threshold trading rules.

Tune thresholds only on validation data.

Use these rules to compare against the RL trading layer.

### Phase 11: Reinforcement Learning Agent

Train RL trading agents that use:

- Market state
- Model probabilities
- Signal-state features
- Portfolio state
- Optional HMM regime features

The RL agent chooses trading actions such as short, flat, long, or target position size.

### Phase 12: Backtesting and Evaluation

Compare all 8 main systems under realistic transaction costs and risk constraints.

Also compare against non-ML baselines and run ablation studies.

The final evaluation should determine whether transformers, HMM regime features, and RL decision policies provide real improvement over simpler baselines.

### imports of major libraries and scripts

In [1]:
#autoreload 
%load_ext autoreload
%autoreload 2
# Core Python
import os
from datetime import datetime, timedelta
from pathlib import Path

# Environment variables
from dotenv import load_dotenv

# Data handling
import numpy as np
import pandas as pd
import polars as pl

# Plotting
import plotly.graph_objects as go
import plotly.express as px

# Progress bars
from tqdm.auto import tqdm

# Alpaca market data
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockLatestTradeRequest

# Project paths
from quant_research.utils.paths import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    ALPACA_RAW_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOKS_DIR,
    RESULTS_DIR,
    LOGS_DIR,
    ensure_project_dirs,
)

# Setup
load_dotenv(PROJECT_ROOT / ".env")
ensure_project_dirs()

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Project root:", PROJECT_ROOT)
print("Raw Alpaca data:", ALPACA_RAW_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/rchbeir/Desktop/Quant work/quant_proj
Raw Alpaca data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca
Processed data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed


/Users/rchbeir/Desktop/Quant work/quant_proj/proj/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Now we are testing the Alpaca connection

In [2]:
from quant_research.data.data_downloader import CheckConnection
CheckConnection()

Alpaca connected


### Downloading the actual data

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo

from quant_research.utils.paths import ALPACA_RAW_DIR
from quant_research.data.data_downloader import Download_5_min_Data

symbols = ["NVDA", "QQQ", "SPY"]
# Alpaca expects timezone-aware datetimes.
# This requests roughly 10 years of historical 5-minute data.
# Format: datetime(year, month, day, hour, minute, tzinfo=ZoneInfo("UTC"))
# 14:30 UTC is 6:30 AM Pacific during standard time, which is the U.S. market open.
# The end date is exclusive, so this covers from Jan 2, 2016 up to Jan 2, 2026.

start = datetime(2016, 1, 2, 14, 30, tzinfo=ZoneInfo("UTC"))
end = datetime(2026, 1, 2, 21, 0, tzinfo=ZoneInfo("UTC"))

df = Download_5_min_Data(
    symbols=symbols,
    start=start,
    end=end,
    timeframe="5min",
    directory=ALPACA_RAW_DIR,
)


NVDA: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/NVDA_5min.parquet
Would you like to overwrite it? (y/n):  n


NVDA: skipping.


QQQ: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/QQQ_5min.parquet
Would you like to overwrite it? (y/n):  n


QQQ: skipping.


SPY: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/SPY_5min.parquet
Would you like to overwrite it? (y/n):  n


SPY: skipping.


### Now we have to check for overlapping days and store them

In [9]:
from quant_research.data.data_processor import save_overlapping_days
save_overlapping_days(symbols)

Transformed NVDA to pacific time
outside candles removed
NVDA has 2515 total days after time filtering
NVDA has 1610 complete days
Transformed QQQ to pacific time
outside candles removed
QQQ has 2515 total days after time filtering
QQQ has 2203 complete days
Transformed SPY to pacific time
outside candles removed
SPY has 2515 total days after time filtering
SPY has 2462 complete days
initial cleaning complete, there are 1539 in common


NVDA: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet
Would you like to save processed data it? (y/n):  y


NVDA: proceeding.
Saved processed NVDA: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet


QQQ: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet
Would you like to save processed data it? (y/n):  y


QQQ: proceeding.
Saved processed QQQ: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet


SPY: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet
Would you like to save processed data it? (y/n):  y


SPY: proceeding.
Saved processed SPY: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet


In [12]:
#quick sanity check to see if they have all the same number of row
for symbol in symbols:
    path = PROCESSED_DATA_DIR / f"{symbol}_5min.parquet"
    df = pd.read_parquet(path)
    print(symbol)
    print("shape:", df.shape)
    print("unique days:", df["date"].nunique())
    print("start:", df["timestamp"].min())
    print("end:", df["timestamp"].max())
    print()
df.head()

NVDA
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

QQQ
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

SPY
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00



,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,timestamp_pt,date,time
0,SPY,2016-02-18 12:45:00+00:00,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00
1,SPY,2016-02-18 12:50:00+00:00,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00
2,SPY,2016-02-18 12:55:00+00:00,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00
3,SPY,2016-02-18 13:00:00+00:00,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00
4,SPY,2016-02-18 13:05:00+00:00,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00
